In [1]:
import polars as pl

from mybookrec import DATA_DIR

In [2]:
# Schemas contracts for transformed data.
BOOKS_SCHEMA = pl.Struct(
    {
        "book_id": pl.Int64,
        "num_pages": pl.Int64,
        "average_rating": pl.Float64,
        "ratings_count": pl.Int64,
        "description": pl.String,
        "title": pl.String,
    }
)

INTERACTIONS_SCHEMA = pl.Struct(
    {
        "user_id": pl.Int64,
        "book_id": pl.Int64,
        "rating": pl.Float64,
        "date_added": pl.Datetime,
        "data_split": pl.String,
    }
)

MY_BOOKS_SCHEMA = pl.Struct(
    {
        "book_id": pl.Int64,
        "my_rating": pl.Float64,
        "average_rating": pl.Float64,
    }
)

GENRES_SCHEMA = pl.Struct(
    {
        "book_id": pl.Int64,
        "genres": pl.Struct 

    }
)

In [3]:
# Load raw book data and cast to the defined schema, while applying necessary transformations and filters
books_load = pl.scan_ndjson(
    DATA_DIR / "raw" / "goodreads_books.json.gz",
    low_memory=True
).select(
    pl.col("book_id"),
    pl.col("num_pages").cast(pl.Int64, strict=False), # Cast num_pages to int, allowing for nulls if conversion fails.
    pl.col("is_ebook") == True, # Produce boolean column indicating if the book is an ebook or not.
    pl.col("average_rating").cast(pl.Float64, strict=False), # Cast average_rating to float, allowing for nulls if conversion fails.
    pl.col("ratings_count").cast(pl.Int64, strict=False), # Cast ratings_count to int, allowing for nulls if conversion fails.
    pl.col("description"),
    pl.col("title")
).filter(
    pl.col("ratings_count") > 5, # Filter out books with no ratings to avoid skewing the model with unrated books.
)

# Load book genres data to join to the main books dataframe, while selecting only relevant columns and applying necessary transformations.
genres_load = pl.scan_ndjson(
    DATA_DIR / "raw" / "goodreads_book_genres_initial.json.gz",
    low_memory=True
).select(
    pl.col("book_id"),
    pl.col("genres")
)

# Load user interactions data and cast to the defined schema, while applying necessary transformations and filters
interactions_load = pl.scan_ndjson(
    DATA_DIR / "raw" / "goodreads_interactions_dedup.json.gz",
    low_memory=True
).select(
    pl.col("user_id"),
    pl.col("book_id"),
    pl.col("rating").cast(pl.Float64, strict=False), # Cast rating to float, allowing for nulls if conversion fails.
    pl.col("date_added").str.replace_all(r"\s+", " ").str.to_datetime("%a %b %d %H:%M:%S %z %Y", strict=False) # Replace multiple spaces with a single space before parsing, allowing for nulls if parsing fails.
).filter(
    pl.col("rating") > 0 # Filter out interactions with no rating to focus on explicit feedback and avoid skewing the model with implicit interactions.
)

# Load my data and filter to only include books that I have rated, while selecting only relevant columns and applying necessary transformations.
my_books_load = pl.read_csv(
    DATA_DIR / "raw" / "goodreads_library_export.csv"
).select(
    pl.col("Book Id").alias("book_id").cast(pl.Int64, strict=False), # Cast book_id to int, allowing for nulls if conversion fails.
    pl.col("My Rating").alias("my_rating").cast(pl.Float64, strict=False) # Cast my_rating to float, allowing for nulls if conversion fails.
).filter(
    pl.col("my_rating") > 0 # Filter out books that I have not rated to focus on explicit feedback and avoid skewing the model with implicit interactions.
).write_csv(
    DATA_DIR / "transformed" / "my_books.csv"
)

In [ ]:
# Inner join books and genres so we can drop books without genre information, which is important for content-based filtering and to avoid missing data issues in modeling. then save to parquet for later use.
books_with_genres = books_load.join(
    genres_load,
    on="book_id",
    how="inner"
).sink_parquet(
    DATA_DIR / "transformed" / "books_with_genres.parquet"
)

# Inner join books and interactions to focus on books that have user interactions, which is important for collaborative filtering and to avoid skewing the model with books that have no user feedback. then save to parquet for later use.
# Also do a 70/20/10 time aware train/test/validation split on the interactions data to prepare for modeling, while ensuring that all interactions for a given user are in the same split to avoid data leakage and ensure realistic evaluation.
books_with_interactions = books_load.join(
    interactions_load,
    on="book_id",
    how="inner"
).with_columns(
    pl.col("date_added").rank("dense", descending = False).over("user_id").alias("interaction_rank") # Rank interactions for each user by date_added to enable time-aware splitting.
).with_columns(
    pl.when(pl.col("interaction_rank") <= pl.col("interaction_rank").max().over("user_id") * 0.7).then(pl.lit("train")) # Assign interactions to train/test/validation splits based on their rank for each user, use pl.lit to ensure the correct type is assigned to the new column.
    .when(pl.col("interaction_rank") <= pl.col("interaction_rank").max().over("user_id") * 0.9).then(pl.lit("test"))
    .otherwise(pl.lit("validation"))
    .alias("data_split")
).sink_parquet(
    DATA_DIR / "transformed" / "books_with_interactions.parquet"
)

## Notes
- if we split globally users with more entires would dominte the split, so we need per user cumilative fraction
- then this cumalitive fraction preserves temporal order, so earliest 70% train, 20% test and last 10% val 